# 02 — Acquire DGEG trade and sales

Discover and download DGEG petroleum-product trade files plus the long annual sales workbook. Raw files are never modified.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)

from portugal_refining_resilience.sources import discover_download_links, download_file, load_source_manifest, year_from_url
from portugal_refining_resilience.io import sha256_file


In [ ]:
sources = load_source_manifest(ROOT / "config" / "sources.yml")
trade_links = discover_download_links(sources["dgeg_trade"]["landing_page"])
sales_links = discover_download_links(sources["dgeg_sales"]["landing_page"])
print(f"DGEG trade downloads discovered: {len(trade_links)}")
print(f"DGEG sales downloads discovered: {len(sales_links)}")


In [ ]:
records = []
for url in trade_links:
    year = year_from_url(url)
    if year is None or year < 2000:
        continue
    suffix = ".xlsx" if ".xlsx" in url.lower() else ".xls"
    target = PATHS.raw / "dgeg" / "trade" / f"dgeg_trade_{year}{suffix}"
    download_file(url, target)
    records.append({"dataset": "dgeg_trade", "year": year, "path": str(target.relative_to(ROOT)), "url": url, "sha256": sha256_file(target)})

# Prefer the long sales workbook when present.
for url in sales_links:
    if "1970" not in url and "2024" not in url:
        continue
    suffix = ".xlsx" if ".xlsx" in url.lower() else ".xls"
    target = PATHS.raw / "dgeg" / f"dgeg_sales_long{suffix}"
    download_file(url, target)
    records.append({"dataset": "dgeg_sales", "year": None, "path": str(target.relative_to(ROOT)), "url": url, "sha256": sha256_file(target)})
    break

manifest = pd.DataFrame(records)
display(manifest.tail())
persist_dataframe(manifest, PATHS.provenance / "dgeg_download_manifest.csv")
